In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

# 1. CONFIG

GPT2_SMALL_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embed_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": True,
}


# 2. TOKENIZER (tiktoken)

def get_tokenizer():
    """Returns GPT-2 tokenizer via tiktoken."""
    return tiktoken.get_encoding("gpt2")


# 3. DATASET & DATALOADER

class TextDataset(Dataset):
    """Sliding window dataset that tokenizes raw text with tiktoken."""

    def __init__(self, text: str, tokenizer, max_length: int, stride: int):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i          : i + max_length]
            target_chunk = token_ids[i + 1      : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(text: str, batch_size: int = 4, max_length: int = 256,
                      stride: int = 128, shuffle: bool = True,
                      drop_last: bool = True, num_workers: int = 0):
    tokenizer = get_tokenizer()
    dataset   = TextDataset(text, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)


# 4. MULTI-HEAD CAUSAL SELF-ATTENTION

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, context_length: int,
                 dropout: float, qkv_bias: bool = False):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"

        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.embed_dim = embed_dim

        self.W_qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=qkv_bias)
        self.W_out = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.W_qkv(x)                          # (B, T, 3C)
        q, k, v = qkv.split(self.embed_dim, dim=-1)  # each (B, T, C)

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Scaled dot-product attention
        scale  = self.head_dim ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale    # (B, n_heads, T, T)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_drop(weights)

        out = weights @ v                              # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)
        return self.resid_drop(out)


# 5. FEED-FORWARD NETWORK

class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# 6. TRANSFORMER BLOCK


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.attn = MultiHeadAttention(
            embed_dim=cfg["embed_dim"],
            n_heads=cfg["n_heads"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff   = FeedForward(cfg["embed_dim"], cfg["drop_rate"])
        self.ln1  = nn.LayerNorm(cfg["embed_dim"])
        self.ln2  = nn.LayerNorm(cfg["embed_dim"])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # pre-LayerNorm
        x = x + self.ff(self.ln2(x))
        return x


# 7. GPT-2 MODEL

class GPTModel(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["embed_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.blocks   = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.ln_f     = nn.LayerNorm(cfg["embed_dim"])
        self.head     = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)


        self.head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        device = idx.device

        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=device))
        x   = self.drop_emb(tok + pos)
        x   = self.blocks(x)
        x   = self.ln_f(x)
        return self.head(x)                            # (B, T, vocab_size)


# 8. TEXT GENERATION UTILITY

@torch.no_grad()
def generate(model, idx, max_new_tokens: int, context_size: int,
             temperature: float = 1.0, top_k: int = None):
    """Autoregressive token generation with temperature and top-k sampling."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        logits   = model(idx_cond)[:, -1, :]          # (B, vocab_size)

        if temperature != 1.0:
            logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        probs    = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat([idx, idx_next], dim=1)
    return idx


def text_to_ids(text: str, tokenizer, device):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)


def ids_to_text(ids: torch.Tensor, tokenizer):
    return tokenizer.decode(ids.squeeze(0).tolist())



# QUICK SMOKE-TEST


if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    cfg   = GPT2_SMALL_CONFIG
    model = GPTModel(cfg).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total:,}")

    tokenizer = get_tokenizer()
    sample    = "The transformer architecture revolutionized NLP"
    ids       = text_to_ids(sample, tokenizer, device)
    out       = model(ids)
    print(f"Forward pass output shape: {out.shape}")   # (1, T, 50257)

    generated = generate(model, ids, max_new_tokens=20,
                         context_size=cfg["context_length"],
                         temperature=0.8, top_k=40)
    print("Generated:", ids_to_text(generated, tokenizer))
    print("Stage 1 complete.")

Device: cuda
Model parameters: 124,439,808
Forward pass output shape: torch.Size([1, 7, 50257])
Generated: The transformer architecture revolutionized NLP Publishersclusive 295 caffe Christian ge undergrad searchingzl sustained Pack asserted bought046 international trauma international Tests Pablo depth
Stage 1 complete.


In the colab terminal run the following Commands



```
sudo apt install zstd
curl -fsSL https://ollama.com/install.sh | sh
ollama serve &
```



In [ ]:
"""
Stage 2: Training Loop, Model Evaluation, Load GPT-2 Small Pretrained Weights
- AdamW optimiser with cosine LR schedule
- Cross-entropy loss + perplexity
- Load GPT-2 Small weights from HuggingFace
- Save / load checkpoints
- Evaluate with Ollama
"""

import os
import math
import json
import time
import torch
import torch.nn as nn
import numpy as np
import tiktoken
import requests
from torch.utils.data import DataLoader


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ──────────────────────────────────────────────
# 1. LOSS HELPERS
# ──────────────────────────────────────────────

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch  = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)                         # (B, T, V)
    loss   = nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()   # (B*T, V), (B*T,)
    )
    return loss


@torch.no_grad()
def calc_loss_loader(data_loader, model, device, num_batches=None):
    model.eval()
    total_loss, count = 0.0, 0
    for i, (x, y) in enumerate(data_loader):
        if num_batches is not None and i >= num_batches:
            break
        total_loss += calc_loss_batch(x, y, model, device).item()
        count += 1
    return total_loss / max(count, 1)


def perplexity(loss: float) -> float:
    return math.exp(loss)


# ──────────────────────────────────────────────
# 2. COSINE LR SCHEDULE
# ──────────────────────────────────────────────

def get_lr(step: int, warmup_steps: int, max_steps: int,
           max_lr: float, min_lr: float) -> float:
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# ──────────────────────────────────────────────
# 3. TRAINING LOOP
# ──────────────────────────────────────────────

def train_model(
    model,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int = 6,
    eval_freq: int = 200,
    eval_iters: int = 20,
    save_path: str = "gpt2_checkpoint.pth",
    max_lr: float = 6e-4,
    min_lr: float = 6e-5,
    warmup_steps: int = 200,
    grad_clip: float = 1.0,
):
    model.to(DEVICE)
    optimiser   = torch.optim.AdamW(model.parameters(), lr=max_lr,
                                     betas=(0.9, 0.95), weight_decay=0.1)
    tokenizer   = get_tokenizer()
    global_step = 0
    history     = {"train_loss": [], "val_loss": [], "steps": []}

    total_steps = n_epochs * len(train_loader)
    print(f"Training for {n_epochs} epoch(s) | {total_steps} total steps")
    print(f"Device: {DEVICE}\n")

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (x, y) in enumerate(train_loader):
            # LR schedule
            lr = get_lr(global_step, warmup_steps, total_steps, max_lr, min_lr)
            for pg in optimiser.param_groups:
                pg["lr"] = lr

            optimiser.zero_grad()
            loss = calc_loss_batch(x, y, model, DEVICE)
            loss.backward()

            # Gradient clipping
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimiser.step()

            epoch_loss  += loss.item()
            global_step += 1

            # Periodic evaluation
            if global_step % eval_freq == 0 or global_step == 1:
                train_loss = calc_loss_loader(train_loader, model, DEVICE, eval_iters)
                val_loss   = calc_loss_loader(val_loader,   model, DEVICE, eval_iters)
                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)
                history["steps"].append(global_step)
                model.train()
                print(
                    f"Ep {epoch+1}/{n_epochs} | Step {global_step:>6} | "
                    f"LR {lr:.2e} | Train loss {train_loss:.4f} "
                    f"(PPL {perplexity(train_loss):.1f}) | "
                    f"Val loss {val_loss:.4f} (PPL {perplexity(val_loss):.1f})"
                )

        avg_epoch = epoch_loss / len(train_loader)
        print(f"\n── End of epoch {epoch+1}: avg train loss = {avg_epoch:.4f} ──\n")

    # Save checkpoint
    save_checkpoint(model, optimiser, global_step, history, save_path)
    return history


# ──────────────────────────────────────────────
# 4. CHECKPOINT SAVE / LOAD
# ──────────────────────────────────────────────

def save_checkpoint(model, optimiser, step, history, path: str):
    checkpoint = {
        "model_state":     model.state_dict(),
        "optimiser_state": optimiser.state_dict(),
        "step":            step,
        "history":         history,
        "config":          GPT2_SMALL_CONFIG,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved → {path}")


def load_checkpoint(path: str, device=DEVICE):
    checkpoint = torch.load(path, map_location=device)
    model = GPTModel(checkpoint["config"]).to(device)
    model.load_state_dict(checkpoint["model_state"])
    print(f"Checkpoint loaded ← {path}  (step {checkpoint['step']})")
    return model, checkpoint


# ──────────────────────────────────────────────
# 5. LOAD GPT-2 SMALL PRETRAINED WEIGHTS
# ──────────────────────────────────────────────

def load_gpt2_pretrained(model: GPTModel) -> GPTModel:
    """
    Transfer GPT-2 Small weights from HuggingFace transformers into our model.
    Requires: pip install transformers
    """
    try:
        from transformers import GPT2Model as HF_GPT2
    except ImportError:
        raise ImportError("Run: pip install transformers")

    print("Downloading GPT-2 Small pretrained weights…")
    hf_model = HF_GPT2.from_pretrained("gpt2")
    hf_sd    = hf_model.state_dict()

    our_sd   = model.state_dict()

    # ── Embedding layers ──
    our_sd["tok_emb.weight"].copy_(hf_sd["wte.weight"])
    our_sd["pos_emb.weight"].copy_(hf_sd["wpe.weight"])

    # ── Transformer blocks ──
    for layer_idx in range(GPT2_SMALL_CONFIG["n_layers"]):
        hf_pfx = f"h.{layer_idx}"
        ou_pfx = f"blocks.{layer_idx}"

        # Layer norms
        our_sd[f"{ou_pfx}.ln1.weight"].copy_(hf_sd[f"{hf_pfx}.ln_1.weight"])
        our_sd[f"{ou_pfx}.ln1.bias"  ].copy_(hf_sd[f"{hf_pfx}.ln_1.bias"])
        our_sd[f"{ou_pfx}.ln2.weight"].copy_(hf_sd[f"{hf_pfx}.ln_2.weight"])
        our_sd[f"{ou_pfx}.ln2.bias"  ].copy_(hf_sd[f"{hf_pfx}.ln_2.bias"])

        # QKV weight and bias (HF stores them as separate q/k/v, GPT-2 as c_attn)
        c_attn_w = hf_sd[f"{hf_pfx}.attn.c_attn.weight"].T  # (C, 3C) → (3C, C)
        c_attn_b = hf_sd[f"{hf_pfx}.attn.c_attn.bias"]
        our_sd[f"{ou_pfx}.attn.W_qkv.weight"].copy_(c_attn_w)
        our_sd[f"{ou_pfx}.attn.W_qkv.bias"  ].copy_(c_attn_b)

        # Output projection
        our_sd[f"{ou_pfx}.attn.W_out.weight"].copy_(
            hf_sd[f"{hf_pfx}.attn.c_proj.weight"].T)
        our_sd[f"{ou_pfx}.attn.W_out.bias"].copy_(
            hf_sd[f"{hf_pfx}.attn.c_proj.bias"])

        # MLP
        our_sd[f"{ou_pfx}.ff.net.0.weight"].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_fc.weight"].T)
        our_sd[f"{ou_pfx}.ff.net.0.bias"  ].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_fc.bias"])
        our_sd[f"{ou_pfx}.ff.net.2.weight"].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_proj.weight"].T)
        our_sd[f"{ou_pfx}.ff.net.2.bias"  ].copy_(
            hf_sd[f"{hf_pfx}.mlp.c_proj.bias"])

    # ── Final LayerNorm ──
    our_sd["ln_f.weight"].copy_(hf_sd["ln_f.weight"])
    our_sd["ln_f.bias"  ].copy_(hf_sd["ln_f.bias"])

    model.load_state_dict(our_sd)
    print("GPT-2 Small weights loaded successfully.")
    return model


# ──────────────────────────────────────────────
# 6. EVALUATE USING OLLAMA
# ──────────────────────────────────────────────

def evaluate_with_ollama(
    model,
    prompts: list[str],
    reference_answers: list[str] | None = None,
    ollama_model: str = "llama3",
    ollama_url: str = "http://localhost:11434/api/generate",
    temperature: float = 0.7,
    top_k: int = 40,
    max_new_tokens: int = 200,
):
    """
    Generate answers from our model and optionally send them to Ollama
    for quality scoring against reference answers.

    Requirements: Ollama running locally  →  ollama serve
    """
    tokenizer = get_tokenizer()
    results   = []

    for idx, prompt in enumerate(prompts):
        print(f"\n── Prompt {idx+1}: {prompt[:80]}…")

        # Our model generates a response
        ids      = text_to_ids(prompt, tokenizer, DEVICE)
        gen_ids  = generate(model, ids, max_new_tokens=max_new_tokens,
                            context_size=GPT2_SMALL_CONFIG["context_length"],
                            temperature=temperature, top_k=top_k)
        our_resp = ids_to_text(gen_ids, tokenizer)
        print(f"Our model: {our_resp[:300]}")

        # Optionally ask Ollama to judge quality
        ollama_score = None
        if reference_answers and idx < len(reference_answers):
            judge_prompt = (
                f"Reference answer:\n{reference_answers[idx]}\n\n"
                f"Model answer:\n{our_resp}\n\n"
                "Rate the model answer for accuracy and relevance (1-10). "
                "Reply with just a JSON object: {\"score\": <int>, \"reason\": \"<short reason>\"}"
            )
            try:
                payload  = {"model": ollama_model, "prompt": judge_prompt,
                            "stream": False}
                response = requests.post(ollama_url, json=payload, timeout=60)
                raw      = response.json().get("response", "")
                parsed   = json.loads(raw)
                ollama_score = parsed
                print(f"Ollama score: {ollama_score}")
            except Exception as e:
                print(f"Ollama eval skipped ({e})")

        results.append({
            "prompt":       prompt,
            "our_response": our_resp,
            "ollama_score": ollama_score,
        })

    return results



# DEMO ENTRY POINT
# ──────────────────────────────────────────────

if __name__ == "__main__":
    CHECKPOINT = "gpt2_checkpoint.pth"

    # ── Build or load model ──
    if os.path.exists(CHECKPOINT):
        model, ckpt = load_checkpoint(CHECKPOINT)
    else:
        model = GPTModel(GPT2_SMALL_CONFIG).to(DEVICE)
        model = load_gpt2_pretrained(model)

        # ── Dummy training data (replace with real corpus) ──
        SAMPLE_TEXT = (
            "Transformers are a type of neural network architecture that uses "
            "self-attention mechanisms. They were introduced in the paper "
            "'Attention Is All You Need' by Vaswani et al. in 2017. "
            "The architecture consists of encoder and decoder stacks, each "
            "containing multi-head attention and feed-forward layers. "
            "GPT models use only the decoder portion of the transformer. "
        ) * 400  # replicate for demo purposes

        train_loader = create_dataloader(SAMPLE_TEXT, batch_size=4,
                                         max_length=256, stride=128, shuffle=True)
        val_loader   = create_dataloader(SAMPLE_TEXT, batch_size=4,
                                         max_length=256, stride=256, shuffle=False)

        history = train_model(
            model, train_loader, val_loader,
            n_epochs=2, eval_freq=50,
            save_path=CHECKPOINT,
        )

    # ── Quick generation test ──
    tokenizer = get_tokenizer()
    prompt    = "Attention mechanisms in neural networks"
    ids       = text_to_ids(prompt, tokenizer, DEVICE)
    out_ids   = generate(model, ids, max_new_tokens=60,
                         context_size=GPT2_SMALL_CONFIG["context_length"],
                         temperature=0.8, top_k=40)
    print("\nGenerated text:")
    print(ids_to_text(out_ids, tokenizer))

    # ── Ollama evaluation
    eval_prompts = [
        "What is the attention mechanism in transformers?",
        "Explain GPT-2 architecture briefly.",
    ]
    evaluate_with_ollama(model, eval_prompts, max_new_tokens=100)
    print("\nStage 2 complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 Small weights loaded successfully.
Training for 2 epoch(s) | 118 total steps
Device: cuda

Ep 1/2 | Step      1 | LR 0.00e+00 | Train loss 1.2688 (PPL 3.6) | Val loss 1.2689 (PPL 3.6)
Ep 1/2 | Step     50 | LR 1.47e-04 | Train loss 0.0095 (PPL 1.0) | Val loss 0.0094 (PPL 1.0)

── End of epoch 1: avg train loss = 0.3539 ──

Ep 2/2 | Step    100 | LR 2.97e-04 | Train loss 0.0025 (PPL 1.0) | Val loss 0.0027 (PPL 1.0)

── End of epoch 2: avg train loss = 0.0176 ──

Checkpoint saved → gpt2_checkpoint.pth

Generated text:
Attention mechanisms in neural networks. They were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture consists of encoder and decoder stacks, each containing multi-head attention and feed-forward layers. GPT models use only the decoder portion of the transformer

── Prompt 1: What is the attention mechanism in transformers?…
Our model: What is the attention mechanism in transformers? Transformers are a type of neural networ

In [ ]:
"""
Stage 3: Fine-tune for Research Paper Personal Assistant
- PDF text extraction (PyMuPDF)
- Build instruction-tuned dataset from paper (summary + Q&A pairs)
- Supervised fine-tuning (SFT) with prompt templates
- Interactive assistant: choose summary or ask questions
- Save fine-tuned model; evaluate with Ollama
"""

import os
import re
import json
import torch
import torch.nn as nn
import tiktoken
import requests
from torch.utils.data import Dataset, DataLoader


FINETUNE_CHECKPOINT = "gpt2_finetuned.pth"
BASE_CHECKPOINT     = "gpt2_checkpoint.pth"


# ──────────────────────────────────────────────
# 1. PDF INGESTION
# ──────────────────────────────────────────────

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract raw text from a PDF using PyMuPDF.
    Install: pip install pymupdf
    """
    try:
        import fitz  # PyMuPDF
    except ImportError:
        raise ImportError("Run: pip install pymupdf")

    doc  = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text.strip()


def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> list[str]:
    """Split text into overlapping chunks for context windows."""
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


# ──────────────────────────────────────────────
# 2. INSTRUCTION DATASET BUILDER
# ──────────────────────────────────────────────

PROMPT_TEMPLATE = (
    "<|context|>\n{context}\n"
    "<|instruction|>\n{instruction}\n"
    "<|response|>\n{response}<|endoftext|>"
)


def build_finetuning_data(paper_text: str) -> list[dict]:
    """
    Builds instruction-response pairs from a research paper.
    In production, replace with an automated pipeline or curated dataset.
    """
    chunks = chunk_text(paper_text, chunk_size=600, overlap=80)
    data   = []

    for i, chunk in enumerate(chunks[:50]):           # limit for demo
        # Summary instruction
        data.append({
            "context":     chunk,
            "instruction": "Summarise the above passage from the research paper.",
            "response":    f"[SUMMARY of chunk {i+1}]: {chunk[:200]}…",
        })
        # Question-answer instruction
        data.append({
            "context":     chunk,
            "instruction": "What is the main contribution described in this passage?",
            "response":    f"[ANSWER]: Based on the passage, the key idea is: {chunk[:150]}…",
        })

    return data


def format_sample(sample: dict) -> str:
    return PROMPT_TEMPLATE.format(**sample)


# ──────────────────────────────────────────────
# 3. FINE-TUNING DATASET
# ──────────────────────────────────────────────

class InstructionDataset(Dataset):
    def __init__(self, samples: list[dict], tokenizer, max_length: int = 512):
        self.pairs = []
        for s in samples:
            text = format_sample(s)
            ids  = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
            if len(ids) > max_length:
                ids = ids[:max_length]
            ids_t = torch.tensor(ids)
            self.pairs.append((ids_t[:-1], ids_t[1:]))   # input, target

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def collate_fn(batch):
    """Pad sequences in a batch to the same length."""
    inputs,  targets = zip(*batch)
    max_len = max(x.size(0) for x in inputs)
    pad_in  = torch.zeros(len(inputs),  max_len, dtype=torch.long)
    pad_tgt = torch.full((len(targets), max_len), -100, dtype=torch.long)
    for i, (inp, tgt) in enumerate(zip(inputs, targets)):
        pad_in [i, :inp.size(0)] = inp
        pad_tgt[i, :tgt.size(0)] = tgt
    return pad_in, pad_tgt


# ──────────────────────────────────────────────
# 4. FINE-TUNING LOOP
# ──────────────────────────────────────────────

def finetune(
    model,
    train_data: list[dict],
    n_epochs: int = 6,
    batch_size: int = 2,
    max_length: int = 512,
    lr: float = 5e-5,
    eval_freq: int = 50,
    save_path: str = FINETUNE_CHECKPOINT,
):
    tokenizer = get_tokenizer()
    dataset   = InstructionDataset(train_data, tokenizer, max_length)
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                           collate_fn=collate_fn)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.to(DEVICE).train()

    total_steps = n_epochs * len(loader)
    step        = 0
    print(f"Fine-tuning: {len(dataset)} samples | {total_steps} steps\n")

    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad()

            logits = model(x)                           # (B, T, V)
            # Flatten; ignore padded positions (target = -100)
            loss = nn.functional.cross_entropy(
                logits.flatten(0, 1), y.flatten(), ignore_index=-100
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()

            epoch_loss += loss.item()
            step       += 1

            if step % eval_freq == 0:
                print(f"Ep {epoch+1}/{n_epochs} | Step {step} | Loss {loss.item():.4f}")

        print(f"\n── Epoch {epoch+1} done | avg loss {epoch_loss/len(loader):.4f} ──\n")

    save_checkpoint(model, optimiser, step, {}, save_path)
    print(f"Fine-tuned model saved → {save_path}")
    return model


# ──────────────────────────────────────────────
# 5. INTERACTIVE ASSISTANT
# ──────────────────────────────────────────────

def build_prompt(context: str, instruction: str) -> str:
    return (
        f"<|context|>\n{context}\n"
        f"<|instruction|>\n{instruction}\n"
        f"<|response|>\n"
    )


@torch.no_grad()
def ask_model(model, context: str, instruction: str,
              max_new_tokens: int = 250, temperature: float = 0.7,
              top_k: int = 40) -> str:
    tokenizer = get_tokenizer()
    prompt    = build_prompt(context, instruction)
    ids       = text_to_ids(prompt, tokenizer, DEVICE)
    out_ids   = generate(
        model, ids, max_new_tokens=max_new_tokens,
        context_size=GPT2_SMALL_CONFIG["context_length"],
        temperature=temperature, top_k=top_k,
    )
    full_text = ids_to_text(out_ids, tokenizer)
    # Extract only the response portion
    marker    = "<|response|>\n"
    if marker in full_text:
        return full_text.split(marker)[-1].split("<|endoftext|>")[0].strip()
    return full_text


def paper_assistant(model, paper_text: str):
    """
    Interactive research paper assistant.
    Asks the user: (1) Summary  or  (2) Ask a specific question.
    """
    chunks  = chunk_text(paper_text, chunk_size=700, overlap=100)
    context = " ".join(chunks[:3])                    # use first ~2100 words

    print("\n" + "═"*60)
    print("  RESEARCH PAPER ASSISTANT")
    print("═"*60)
    print("Paper loaded. What would you like to do?\n")
    print("  [1] Get a summary of the paper")
    print("  [2] Ask a specific question about the paper")
    print("  [q] Quit\n")

    while True:
        choice = input("Your choice: ").strip().lower()

        if choice == "q":
            print("Goodbye!")
            break

        elif choice == "1":
            print("\nGenerating summary…\n")
            summary = ask_model(
                model, context,
                "Please provide a detailed summary of this research paper, "
                "including its main contributions, methodology, and conclusions.",
                max_new_tokens=300,
            )
            print("─"*60)
            print(summary)
            print("─"*60 + "\n")

        elif choice == "2":
            question = input("Your question: ").strip()
            if not question:
                continue
            print("\nSearching for answer…\n")
            answer = ask_model(
                model, context, question,
                max_new_tokens=200,
            )
            print("─"*60)
            print(answer)
            print("─"*60 + "\n")

        else:
            print("Please enter 1, 2 or q.\n")

        print("\nWhat would you like to do next?")
        print("  [1] Summary  [2] Ask a question  [q] Quit\n")


# ──────────────────────────────────────────────
# 6. OLLAMA EVALUATION FOR FINE-TUNED MODEL
# ──────────────────────────────────────────────

def evaluate_finetuned_with_ollama(
    model,
    paper_text: str,
    ollama_model: str = "llama3",
    ollama_url:   str = "http://localhost:11434/api/generate",
):
    """Evaluate our fine-tuned model answers against Ollama's judgment."""
    chunks  = chunk_text(paper_text, chunk_size=700, overlap=100)
    context = " ".join(chunks[:3])

    test_cases = [
        {
            "instruction": "Summarise the main contribution of this paper.",
            "reference":   "A summary should capture the main idea, method, and result.",
        },
        {
            "instruction": "What dataset was used in the experiments?",
            "reference":   "The answer should mention any datasets described in the paper.",
        },
        {
            "instruction": "What limitations does the paper acknowledge?",
            "reference":   "The answer should describe limitations or future work.",
        },
    ]

    scores = []
    for tc in test_cases:
        response = ask_model(model, context, tc["instruction"])
        print(f"\nQ: {tc['instruction']}")
        print(f"A: {response[:300]}")

        judge_prompt = (
            f"Reference expectation: {tc['reference']}\n"
            f"Model answer: {response}\n\n"
            "Score the model answer 1-10 for relevance and accuracy. "
            "Reply ONLY with JSON: {\"score\": <int>, \"reason\": \"<brief>\"}"
        )
        try:
            r      = requests.post(ollama_url,
                                   json={"model": ollama_model,
                                         "prompt": judge_prompt, "stream": False},
                                   timeout=60)
            parsed = json.loads(r.json().get("response", "{}"))
            print(f"Ollama score: {parsed}")
            scores.append(parsed.get("score", 0))
        except Exception as e:
            print(f"Ollama unavailable: {e}")

    if scores:
        print(f"\nAverage Ollama score: {sum(scores)/len(scores):.1f}/10")


# ──────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────

if __name__ == "__main__":
    import sys

    # ── Load base model ──
    if os.path.exists(FINETUNE_CHECKPOINT):
        model, _ = load_checkpoint(FINETUNE_CHECKPOINT)
        print("Loaded fine-tuned model.")
    elif os.path.exists(BASE_CHECKPOINT):
        model, _ = load_checkpoint(BASE_CHECKPOINT)
        print("Loaded base checkpoint for fine-tuning.")
    else:
        print("No checkpoint found. Run stage2_training.py first.")
        sys.exit(1)

    # ── Load paper ──
    PDF_PATH = sys.argv[1] if len(sys.argv) > 1 else None

    if PDF_PATH and os.path.exists(PDF_PATH):
        print(f"Loading PDF: {PDF_PATH}")
        paper_text = extract_text_from_pdf(PDF_PATH)
    else:
        # Fallback: demo text
        print("No PDF provided. Using demo paper text.")
        paper_text = (
            "Abstract: We present a novel approach to natural language understanding "
            "using large-scale transformer models pre-trained on diverse corpora. "
            "Our method achieves state-of-the-art results on multiple benchmarks "
            "including GLUE and SuperGLUE. We introduce a new training objective "
            "that combines masked language modelling with next-sentence prediction. "
            "Experiments on datasets including BookCorpus, Wikipedia, and Common Crawl "
            "demonstrate significant improvements over prior work. The model achieves "
            "89.2 on GLUE, surpassing previous methods by 2.1 points. Limitations "
            "include high computational requirements and potential biases in training data. "
            "Future work will explore parameter-efficient fine-tuning methods."
        ) * 30

    # ── Fine-tune if not already done ──
    if not os.path.exists(FINETUNE_CHECKPOINT):
        print("\nBuilding fine-tuning dataset from paper…")
        train_data = build_finetuning_data(paper_text)
        print(f"Created {len(train_data)} training samples.\n")
        model = finetune(model, train_data, n_epochs=3, batch_size=2)

    # ── Ollama evaluation ──
    print("\nRunning Ollama evaluation…")
    evaluate_finetuned_with_ollama(model, paper_text)

    # ── Interactive assistant ──
    paper_assistant(model, paper_text)

Checkpoint loaded ← gpt2_checkpoint.pth  (step 118)
Loaded base checkpoint for fine-tuning.
No PDF provided. Using demo paper text.

Building fine-tuning dataset from paper…
Created 12 training samples.

Fine-tuning: 12 samples | 18 steps


── Epoch 1 done | avg loss 3.0295 ──


── Epoch 2 done | avg loss 1.0311 ──


── Epoch 3 done | avg loss 0.5910 ──

Checkpoint saved → gpt2_finetuned.pth
Fine-tuned model saved → gpt2_finetuned.pth

Running Ollama evaluation…

Q: Summarise the main contribution of this paper.
A: Abstract: We present a novel approach to natural language understanding using large-scale transformer models pre-trained on diverse corpora. Our method achieves state-of-the-art results on multiple benchmarks including GLUE and SuperGLUE. We introduce a new training objective that combines masked la
Ollama score: {}

Q: What dataset was used in the experiments?
A: What dataset was used in the experiments?
<|context|>
What dataset was used in the experiments?
<|parameter-effi